**CI twin of `ch15-imbalanced-data.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, recall_score,
                             precision_score, confusion_matrix)
import pandas as pd

df = load_csv("penguins").dropna(subset=["flipper_length_mm",
                                         "body_mass_g"])
feats = ["flipper_length_mm", "body_mass_g"]

common = df[df["species"] != "Chinstrap"]
rare = df[df["species"] == "Chinstrap"].sample(n=20, random_state=7)
data = pd.concat([common, rare])
y = (data["species"] == "Chinstrap").astype(int)
print(f"{len(common)} common vs {len(rare)} rare "
      f"({y.mean():.1%} rare)")

Xtr, Xte, ytr, yte = train_test_split(
    data[feats], y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

untreated = LogisticRegression(max_iter=1000).fit(Xtr_s, ytr)
pred = untreated.predict(Xte_s)
print(f"\naccuracy:          {accuracy_score(yte, pred):.3f}")
print(f"rare-class recall: {recall_score(yte, pred, zero_division=0):.3f}")
print(confusion_matrix(yte, pred))

In [ ]:
balanced = LogisticRegression(max_iter=1000,
                              class_weight="balanced").fit(Xtr_s, ytr)
pred_b = balanced.predict(Xte_s)

print(f"accuracy:          {accuracy_score(yte, pred_b):.3f}")
print(f"rare-class recall: {recall_score(yte, pred_b):.3f}")
print(f"precision:         {precision_score(yte, pred_b):.3f}")
print(confusion_matrix(yte, pred_b))

In [ ]:
rare_train_idx = ytr[ytr == 1].index
copies = (ytr == 0).sum() // len(rare_train_idx)   # to rough parity

over_X = pd.concat([Xtr] + [Xtr.loc[rare_train_idx]] * copies)
over_y = pd.concat([ytr] + [ytr.loc[rare_train_idx]] * copies)
print(f"training set: {len(ytr)} rows ({ytr.mean():.1%} rare)  ->  "
      f"{len(over_y)} rows ({over_y.mean():.1%} rare)")

sc_o = StandardScaler().fit(over_X)
over_model = LogisticRegression(max_iter=1000).fit(
    sc_o.transform(over_X), over_y)
pred_o = over_model.predict(sc_o.transform(Xte))
print(f"recall {recall_score(yte, pred_o):.3f}   "
      f"precision {precision_score(yte, pred_o):.3f}")

In [ ]:
# WRONG — for demonstration only.
rare_all_idx = y[y == 1].index
copies_all = (y == 0).sum() // len(rare_all_idx)
leak_X = pd.concat([data[feats]] + [data[feats].loc[rare_all_idx]] * copies_all)
leak_y = pd.concat([y] + [y.loc[rare_all_idx]] * copies_all)

a, b, c, d = train_test_split(leak_X, leak_y, test_size=0.25,
                              random_state=42, stratify=leak_y)
sc_l = StandardScaler().fit(a)
leak_model = LogisticRegression(max_iter=1000).fit(sc_l.transform(a), c)
pred_l = leak_model.predict(sc_l.transform(b))
print(f"'test' recall {recall_score(d, pred_l):.3f}   "
      f"'test' precision {precision_score(d, pred_l):.3f}")

overlap = sum(1 for i in b.index if i in a.index)
print(f"\ntest rows that ALSO appear in training: {overlap} of {len(b)}")

In [ ]:
from sklearn.metrics import roc_auc_score

probs = untreated.predict_proba(Xte_s)[:, 1]
print(f"untreated model's AUC: {roc_auc_score(yte, probs):.3f}\n")

print("threshold   recall   precision   birds flagged")
for t in (0.5, 0.15, 0.08, 0.04):
    p = (probs >= t).astype(int)
    print(f"   {t:<9}{recall_score(yte, p, zero_division=0):.3f}     "
          f"{precision_score(yte, p, zero_division=0):.3f}        "
          f"{int(p.sum())}")

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(Xtr_s, ytr)
pred = model.predict(Xte_s)

run_tests([
    ("every rare bird found", round(recall_score(yte, pred), 3), 1.0),
    ("the price, in precision", round(precision_score(yte, pred), 3), 0.152),
])

In [ ]:
from collections import Counter

def balanced_weights(labels):
    counts = Counter(labels)
    n, k = len(labels), len(counts)
    return {cls: n / (k * c) for cls, c in counts.items()}

def oversample_indices(labels):
    minority = [i for i, v in enumerate(labels) if v == 1]
    majority_count = sum(1 for v in labels if v == 0)
    extras = majority_count - len(minority)
    out = list(range(len(labels)))
    out += [minority[i % len(minority)] for i in range(extras)]
    return out

run_tests([
    ("four to one", balanced_weights([0, 0, 0, 0, 1]), {0: 0.625, 1: 2.5}),
    ("already balanced", balanced_weights([0, 1, 0, 1]), {0: 1.0, 1: 1.0}),
    ("one rare row, copied to parity",
     oversample_indices([0, 0, 0, 1]), [0, 1, 2, 3, 3, 3]),
    ("two rare rows, cycling",
     oversample_indices([0, 0, 0, 0, 1, 1]), [0, 1, 2, 3, 4, 5, 4, 5]),
])